# 00 · Setup: environment, data, checkpoints

Every result in the paper comes from a run in the campaign registry (`repro/campaign.json`), e.g. `r0_s0`
(the undefended predictor, seed 0) or `e1_R3_s1` (attacks on the patched predictor R3, seed 1).
For each run the repo keeps the exact configuration it used (`repro/configs/<key>.yaml`), and the GitHub
release **`checkpoints-v1`** holds every model those configurations need (195 files, 19.3 GB; list with
sha256 in `repro/checkpoints.json`) plus the small per-run outputs (`run_results.tar.gz`, 74 MB).

| notebook | what it produces | needs |
|---|---|---|
| `10_train_predictors` | R0 (undefended predictor) and the three one-shot baselines | GPU, training |
| `11_train_patch_loops` | the attack → detect → patch loops (R1–R3), ablations, multi-space loop | GPU, training |
| `12_train_attackers` | fresh attackers of E1, attacker-seed control, benchmark attackers, ε pilot | GPU, training |
| `13_train_enhancers` | speech enhancer pre-training and GRPO post-training (E3) | GPU, training |
| `20_e2_agreement_table2` | Table 2 (agreement with human ratings), R0 reproduction check | checkpoints |
| `21_e1_hackability_table1` | Table 1 (defences), threshold sweep, attacker-seed variance | checkpoints |
| `22_e3_downstream_table3` | Table 3 (enhancers trained against each predictor) | checkpoints |
| `23_benchmark_published` | benchmark table of published predictors, Fig. panel (a) | checkpoints |
| `24_patch_rounds_figure` | in-loop attack success per patch round, Fig. panel (b) | checkpoints |
| `30_paper_assets` | everything at once + number-provenance check of `paper/main.tex` | checkpoints |

Evaluation notebooks (20–30) never train: attackers, predictors and enhancers are loaded from the release.
Each has `SOURCE="recompute"` (re-run the evaluation on GPUs) or `SOURCE="released"` (only aggregate the original
outputs, CPU, seconds). Run 20 before 21 (Table 1 uses E2 columns) and 23 before 24 (one figure, two panels).

## Environment

```bash
bash scripts/install_kernel.sh   # once: kernel "hackpatch" = project venv + the scripts/env.sh variables
```
Then pick the kernel "hackpatch (scripts/env.sh)" in any Jupyter (e.g. an OnDemand session).
To run a notebook non-interactively: `NB_SOURCE=released jupyter nbconvert --to notebook --execute --inplace notebooks/20_e2_agreement_table2.ipynb`.
Heavy work is sent to Slurm by the notebooks themselves; the kernel only needs a GPU for `RUN="local"`.

In [1]:
import os, sys
from pathlib import Path

# Works when started from the repo root or from notebooks/; needs the project environment
# (`source scripts/env.sh`, kernel "hackpatch" — see 00_setup.ipynb).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
import torch
print("torch", torch.__version__, "| cuda visible:", torch.cuda.is_available())
print("scratch runs dir:", repro.RUNS)
print("artifacts dir   :", repro.ARTIFACTS)

torch 2.10.0+cu128 | cuda visible: False
scratch runs dir: /fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability
artifacts dir   : /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release


## Data

Audio corpora are not redistributed. The manifests (CSV: `utt_id, path, mos, system, corpus, split`) are built by
`scripts/build_eval_manifests.py` (BVCC, NISQA, SOMOS, TMHINT-QI, VMC'23, URGENT) and `scripts/build_vbd_manifests.py`
(VoiceBank-DEMAND for the enhancer); download sources are listed in `docs/eval_corpora.md`.

In [3]:
from hackpatch.data.manifests import manifest_path
needed = ["bvcc/train", "bvcc/val", "bvcc/test", "bvcc_ood/test", "nisqa/test_for", "nisqa/test_p501",
          "nisqa/test_livetalk", "somos/test", "tmhintqi/test", "vmc23/track1a", "vmc23/track1b", "vmc23/track2",
          "vmc23/track3", "urgent24_sqa/test", "urgent25_sqa/test"]
pd.DataFrame([{"manifest": s, "exists": manifest_path(*s.split("/")).exists()} for s in needed])

,manifest,exists
0,bvcc/train,True
1,bvcc/val,True
2,bvcc/test,True
3,bvcc_ood/test,True
4,nisqa/test_for,True
5,nisqa/test_p501,True
6,nisqa/test_livetalk,True
7,somos/test,True
8,tmhintqi/test,True
9,vmc23/track1a,True


## Checkpoints and released outputs

`repro.fetch()` downloads every checkpoint (19.3 GB) and verifies its sha256; on the original cluster it symlinks the
original files instead. Pass registry keys to fetch a subset, e.g. `repro.fetch(["e1_R3_s0"])`. The evaluation
notebooks fetch what they need automatically. The release is private: a GitHub token is read from `GITHUB_TOKEN` or
the git credential store.

In [4]:
ckpts = repro.fetch()
m = repro.manifest()
pd.DataFrame(m["assets"]).groupby("role").agg(files=("asset", "size"), GB=("size", lambda s: round(s.sum() / 1e9, 2)))

[fetch] 195 checkpoints ready in /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints


,files,GB
role,,
attacker,128,0.79
loop_predictor,36,13.59
referenced,3,0.03
selected,28,4.86


In [5]:
runs_dir = repro.fetch_results()
reg = repro.registry()
summary = pd.Series({k: v["kind"] for k, v in reg.items()}).value_counts().rename("runs")
summary

e2                   30
e1                   24
e3_grpo              16
baseline              9
ablation              6
bench                 4
hnp                   3
r0                    3
e3_eval               3
multispace            3
e1_attacker_seeds     3
e3_eval_r2            1
Name: runs, dtype: int64